# UC04 — Puntuación Dinámica de Riesgo en Auto

Motor de puntuación combinando perfil del conductor y datos telemáticos.
**Valor de negocio**: Tarificación dinámica por segmento.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS RIESGO_AUTO_DB;
USE SCHEMA RIESGO_AUTO_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Pólizas y Vehículos

In [ ]:
CREATE OR REPLACE TABLE polizas_auto AS
SELECT 'POL' || LPAD(SEQ4(),5,'0') AS poliza_id, UNIFORM(18,75,RANDOM()) AS edad_conductor,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.5 THEN 'M' ELSE 'F' END AS sexo, UNIFORM(1,40,RANDOM()) AS anos_carnet,
    CASE MOD(UNIFORM(1,100,RANDOM()),3) WHEN 0 THEN 'Turismo' WHEN 1 THEN 'SUV' ELSE 'Furgoneta' END AS tipo_vehiculo,
    UNIFORM(0,20,RANDOM()) AS antiguedad_vehiculo, UNIFORM(5000,60000,RANDOM())::DECIMAL(10,2) AS valor_vehiculo,
    UNIFORM(1,15,RANDOM()) AS bonus_malus,
    CASE WHEN UNIFORM(0,1,RANDOM())<0.35 THEN 1 ELSE 0 END AS ha_tenido_siniestro
FROM TABLE(GENERATOR(ROWCOUNT=>800));

---

## Paso 3: Generar Datos Telemáticos

In [ ]:
CREATE OR REPLACE TABLE telematica AS
SELECT p.poliza_id,
    CASE WHEN p.ha_tenido_siniestro=1 THEN UNIFORM(1200,3000,RANDOM()) ELSE UNIFORM(500,1800,RANDOM()) END AS km_mes_medio,
    CASE WHEN p.ha_tenido_siniestro=1 THEN UNIFORM(15,45,RANDOM()) ELSE UNIFORM(2,20,RANDOM()) END::FLOAT AS pct_conduccion_nocturna,
    CASE WHEN p.ha_tenido_siniestro=1 THEN UNIFORM(8,30,RANDOM()) ELSE UNIFORM(0,10,RANDOM()) END::FLOAT AS pct_exceso_velocidad,
    CASE WHEN p.ha_tenido_siniestro=1 THEN UNIFORM(8,25,RANDOM()) ELSE UNIFORM(1,8,RANDOM()) END AS frenazos_bruscos_mes,
    CASE WHEN p.ha_tenido_siniestro=1 THEN UNIFORM(5,20,RANDOM()) ELSE UNIFORM(0,6,RANDOM()) END AS aceleraciones_bruscas_mes,
    CASE WHEN p.ha_tenido_siniestro=1 THEN UNIFORM(20,55,RANDOM()) ELSE UNIFORM(55,95,RANDOM()) END AS puntuacion_eco
FROM polizas_auto p;

---

## Paso 4: Feature Engineering

In [ ]:
CREATE OR REPLACE TABLE features_riesgo AS
SELECT p.poliza_id, p.edad_conductor::FLOAT AS edad_conductor, p.anos_carnet::FLOAT AS anos_carnet,
    CASE WHEN p.tipo_vehiculo='Turismo' THEN 1.0 ELSE 0.0 END AS es_turismo,
    CASE WHEN p.tipo_vehiculo='SUV' THEN 1.0 ELSE 0.0 END AS es_suv,
    p.antiguedad_vehiculo::FLOAT AS antiguedad_vehiculo, p.valor_vehiculo::FLOAT AS valor_vehiculo, p.bonus_malus::FLOAT AS bonus_malus,
    t.km_mes_medio::FLOAT AS km_mes, t.pct_conduccion_nocturna AS pct_nocturna, t.pct_exceso_velocidad AS pct_exceso,
    t.frenazos_bruscos_mes::FLOAT AS frenazos, t.aceleraciones_bruscas_mes::FLOAT AS aceleraciones, t.puntuacion_eco::FLOAT AS eco_score,
    p.ha_tenido_siniestro
FROM polizas_auto p JOIN telematica t ON p.poliza_id=t.poliza_id;

---

## Paso 5: Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM features_riesgo SAMPLE(80);
CREATE OR REPLACE TABLE test AS SELECT * FROM features_riesgo WHERE poliza_id NOT IN (SELECT poliza_id FROM entrenamiento);

---

## Paso 6: Entrenar Modelo

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION scoring_riesgo_auto(
    INPUT_DATA=>SYSTEM$REFERENCE('TABLE','entrenamiento'),
    TARGET_COLNAME=>'ha_tenido_siniestro',
    CONFIG_OBJECT=>{'evaluate':TRUE}
);

---

## Paso 7: Evaluar

In [ ]:
CALL scoring_riesgo_auto!SHOW_EVALUATION_METRICS();

In [ ]:
CALL scoring_riesgo_auto!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar y Segmentar

In [ ]:
CREATE OR REPLACE TABLE predicciones_riesgo AS
SELECT poliza_id,
    scoring_riesgo_auto!PREDICT(OBJECT_CONSTRUCT(
        'edad_conductor',edad_conductor,'anos_carnet',anos_carnet,'es_turismo',es_turismo,'es_suv',es_suv,
        'antiguedad_vehiculo',antiguedad_vehiculo,'valor_vehiculo',valor_vehiculo,'bonus_malus',bonus_malus,
        'km_mes',km_mes,'pct_nocturna',pct_nocturna,'pct_exceso',pct_exceso,'frenazos',frenazos,
        'aceleraciones',aceleraciones,'eco_score',eco_score
    )) AS prediccion,
    prediccion['probability']['1']::FLOAT AS prob_siniestro,
    CASE WHEN prediccion['probability']['1']::FLOAT>=0.60 THEN 'Prima Alta'
         WHEN prediccion['probability']['1']::FLOAT>=0.30 THEN 'Prima Media'
         ELSE 'Prima Baja' END AS segmento_prima,
    ha_tenido_siniestro AS siniestro_real
FROM test;

SELECT * FROM predicciones_riesgo ORDER BY prob_siniestro DESC LIMIT 20;

---

## Paso 9: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Puntuación Dinámica de Riesgo en Auto')
df = session.table('predicciones_riesgo').to_pandas()

c1,c2,c3 = st.columns(3)
with c1: st.metric('Total Pólizas', len(df))
with c2: st.metric('Prima Alta', len(df[df['SEGMENTO_PRIMA']=='Prima Alta']))
with c3: st.metric('Exactitud', f"{(df['PROB_SINIESTRO'].round()==df['SINIESTRO_REAL']).mean():.1%}")

st.markdown('---')
rc = df['SEGMENTO_PRIMA'].value_counts()
st.bar_chart(pd.DataFrame({'Pólizas': rc.values}, index=rc.index))

st.markdown('---')
filtro = st.multiselect('Segmento', ['Prima Alta','Prima Media','Prima Baja'], default=['Prima Alta','Prima Media','Prima Baja'])
fdf = df[df['SEGMENTO_PRIMA'].isin(filtro)].sort_values('PROB_SINIESTRO', ascending=False)
fdf['Prob %'] = fdf['PROB_SINIESTRO'].apply(lambda x: f'{x:.1%}')
st.dataframe(fdf[['POLIZA_ID','Prob %','SEGMENTO_PRIMA','SINIESTRO_REAL']], use_container_width=True, height=400)
st.caption('Desarrollado con Snowflake Cortex ML + Streamlit')

---

## Paso 10: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS RIESGO_AUTO_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;